In [2]:
!pip install yfinance

StatementMeta(, 6fbeee2b-c867-4a1b-b895-a9a2bedffad0, 4, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 66.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 54.8 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 12.0
    Uninstalling websockets-12.0:
      Successfully uninstalled websockets-12.0
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.


In [3]:
# ============================================================
# NB03_Fundamental_Bronze_Ingestion (ROBUST PRODUCTION FIX)
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import time
import uuid

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# 1. Load the full eligible non-financial universe
universe = spark.sql("""
    SELECT symbol, yahoo_symbol, company_name, industry
    FROM bronze_universe_snapshot
    WHERE instrument_type = 'EQUITY'
      AND eligible_for_stock_screen = true
""").toPandas()

print(f"Total eligible stocks to ingest fundamentals for: {len(universe)}")

# Metric translation dictionary
METRIC_MAP = {
    "Operating Income": "EBIT",
    "Total Revenue": "Total Revenue",
    "Net Income": "Net Income",
    "Diluted EPS": "Diluted EPS",
    "Basic EPS": "Basic EPS",
    "Total Current Assets": "Total Current Assets",
    "Total Current Liabilities": "Total Current Liabilities",
    "Total Debt": "Total Debt",
    "Long Term Debt": "Long Term Debt",
    "Cash And Cash Equivalents": "Cash And Cash Equivalents",
    "Net PPE": "Net PPE",
    "Gross PPE": "Net PPE",
    "Stockholders Equity": "Total Equity",
    "Total Equity": "Total Equity",
    "Common Stock Equity": "Total Equity",
    "Operating Cash Flow": "Operating Cash Flow",
    "Capital Expenditure": "Capital Expenditure",
    "Free Cash Flow": "Free Cash Flow"
}

def safe_float(val):
    try:
        if val is None or pd.isna(val):
            return None
        return float(val)
    except:
        return None


def extract_statement_records(df, stmt_type, period_type, sym, y_sym):
    recs = []

    if df is None or df.empty:
        return recs

    for col_date in df.columns:
        p_str = (
            col_date.strftime("%Y-%m-%d")
            if hasattr(col_date, "strftime")
            else str(col_date)[:10]
        )

        for raw_metric, val in df[col_date].items():
            clean_name = METRIC_MAP.get(str(raw_metric).strip())
            f_val = safe_float(val)

            if clean_name and f_val is not None:
                recs.append({
                    "symbol": sym,
                    "yahoo_symbol": y_sym,
                    "period_end": p_str,
                    "period_type": period_type,
                    "statement_type": stmt_type,
                    "metric_name": clean_name,
                    "metric_value": f_val,
                    "run_id": RUN_ID,
                    "retrieved_at": RUN_TIMESTAMP
                })

    return recs


all_fundamentals = []
all_snapshots = []
failed_stocks = []

# ------------------------------------------------------------
# Fetch fundamentals
# ------------------------------------------------------------

for idx, row in universe.iterrows():

    sym = row["symbol"]
    y_sym = row["yahoo_symbol"]

    if (idx + 1) % 25 == 0 or idx == 0:
        print(
            f"[{idx+1}/{len(universe)}] "
            f"Fetching fundamentals... (Current: {sym})"
        )

    try:
        t = yf.Ticker(y_sym)

        # Pull Annual & Quarterly Statements
        all_fundamentals.extend(
            extract_statement_records(
                t.income_stmt,
                "Income Statement",
                "Annual",
                sym,
                y_sym
            )
        )

        all_fundamentals.extend(
            extract_statement_records(
                t.quarterly_income_stmt,
                "Income Statement",
                "Quarterly",
                sym,
                y_sym
            )
        )

        all_fundamentals.extend(
            extract_statement_records(
                t.balance_sheet,
                "Balance Sheet",
                "Annual",
                sym,
                y_sym
            )
        )

        all_fundamentals.extend(
            extract_statement_records(
                t.quarterly_balance_sheet,
                "Balance Sheet",
                "Quarterly",
                sym,
                y_sym
            )
        )

        all_fundamentals.extend(
            extract_statement_records(
                t.cashflow,
                "Cash Flow",
                "Annual",
                sym,
                y_sym
            )
        )

        all_fundamentals.extend(
            extract_statement_records(
                t.quarterly_cashflow,
                "Cash Flow",
                "Quarterly",
                sym,
                y_sym
            )
        )

        # Fast info & metadata
        info = t.info or {}
        fast = getattr(t, "fast_info", {})

        mcap = (
            safe_float(info.get("marketCap"))
            or safe_float(getattr(fast, "market_cap", None))
        )

        shares = (
            safe_float(info.get("sharesOutstanding"))
            or safe_float(getattr(fast, "shares", None))
        )

        all_snapshots.append({
            "symbol": sym,
            "yahoo_symbol": y_sym,
            "company_name": row["company_name"],
            "sector": info.get("sector", "Unknown"),
            "industry": row["industry"],
            "currency": info.get("currency", "INR"),
            "market_cap": mcap,
            "shares_outstanding": shares,
            "trailing_eps": safe_float(info.get("trailingEps")),
            "book_value": safe_float(info.get("bookValue")),
            "last_updated": RUN_TIMESTAMP,
            "run_id": RUN_ID
        })

    except Exception as e:

        failed_stocks.append({
            "symbol": sym,
            "error": str(e)
        })

        print(f"⚠️ Failed: {sym} | {str(e)[:150]}")

    # Polite delay
    time.sleep(0.3)


print(
    f"\nFetch complete! "
    f"Extracted {len(all_fundamentals):,} fundamental data points."
)

print(
    f"Successful: {len(all_snapshots)} | "
    f"Failed: {len(failed_stocks)}"
)


# ------------------------------------------------------------
# Write to Bronze Delta Tables
# ------------------------------------------------------------

if all_fundamentals:

    df_fund = pd.DataFrame(all_fundamentals)

    spark.createDataFrame(df_fund) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_fundamentals_long")

    print(
        f"✅ Saved: bronze_fundamentals_long "
        f"({len(df_fund):,} rows)"
    )


if all_snapshots:

    df_snap = pd.DataFrame(all_snapshots)

    spark.createDataFrame(df_snap) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_market_snapshot")

    print(
        f"✅ Saved: bronze_market_snapshot "
        f"({len(df_snap):,} rows)"
    )

StatementMeta(, 6fbeee2b-c867-4a1b-b895-a9a2bedffad0, 5, Finished, Available, Finished, False)

Total eligible stocks to ingest fundamentals for: 398
[1/398] Fetching fundamentals... (Current: POWERGRID)
[25/398] Fetching fundamentals... (Current: SCHNEIDER)
[50/398] Fetching fundamentals... (Current: GODREJIND)
[75/398] Fetching fundamentals... (Current: IRB)
[100/398] Fetching fundamentals... (Current: DCMSHRIRAM)
[125/398] Fetching fundamentals... (Current: ERIS)
[150/398] Fetching fundamentals... (Current: BATAINDIA)
[175/398] Fetching fundamentals... (Current: CGPOWER)
[200/398] Fetching fundamentals... (Current: JSWDULUX)
[225/398] Fetching fundamentals... (Current: LGEINDIA)
[250/398] Fetching fundamentals... (Current: MSUMI)
[275/398] Fetching fundamentals... (Current: ONESOURCE)
[300/398] Fetching fundamentals... (Current: AWL)
[325/398] Fetching fundamentals... (Current: ANANTRAJ)
[350/398] Fetching fundamentals... (Current: TATAPOWER)
[375/398] Fetching fundamentals... (Current: UBL)

Fetch complete! Extracted 40,178 fundamental data points.
Successful: 398 | Failed: 0

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DUMMYHEG.NS"}}}


In [ ]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_fundamentals_long
        WHERE symbol = 'RELIANCE'
          AND metric_name IN ('EBIT', 'Total Revenue', 'Net Income', 
                              'Total Current Assets', 'Total Debt', 
                              'Operating Cash Flow')
        ORDER BY period_end DESC
    """)
)

StatementMeta(, 1345157a-14f4-4b74-892c-8a72a85e9055, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_market_snapshot
        WHERE symbol IN ('RELIANCE', 'TCS', 'INFY', 'LT')
        ORDER BY market_cap DESC
    """)
)

StatementMeta(, 1345157a-14f4-4b74-892c-8a72a85e9055, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT
            metric_name,
            COUNT(*) AS total_rows,
            COUNT(metric_value) AS non_null_rows,
            ROUND(AVG(metric_value), 2) AS avg_value
        FROM bronze_fundamentals_long
        WHERE period_type = 'Annual'
          AND period_end >= '2020-01-01'
        GROUP BY metric_name
        ORDER BY total_rows DESC
    """)
)

StatementMeta(, 1345157a-14f4-4b74-892c-8a72a85e9055, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_ingestion_log
        ORDER BY run_timestamp DESC
        LIMIT 5
    """)
)

StatementMeta(, 1345157a-14f4-4b74-892c-8a72a85e9055, -1, Cancelled, , Cancelled, True)